# Data Cleaning & Pre-processing

Data source: [Fixture Downloads](https://fixturedownload.com/results/wsl-2024)

In [1]:
# Packages
import polars as pl
import csv
import sys

In [2]:
data_23 = pl.read_csv("data/wsl-2023-UTC.csv")
data_24 = pl.read_csv("data/wsl-2024-UTC.csv")

print(
    data_23.shape,
    data_24.shape,
    sep='\n'
)

if data_23.columns != data_24.columns:
    sys.exit("Not matching")

(132, 7)
(131, 7)


In [3]:
# Insert the season year as a new column
data_23 = data_23.clone().with_columns(pl.lit("23/24").alias("Season"))
data_24 = data_24.clone().with_columns(pl.lit("24/25").alias("Season"))

# Join the two dataframes on top of each other
data = pl.concat([data_23, data_24])

print(data.shape)

(263, 8)


In [4]:
data.head()

Match Number,Round Number,Date,Location,Home Team,Away Team,Result,Season
i64,i64,str,str,str,str,str,str
1,1,"""01/10/2023 11:30""","""Villa Park""","""Aston Villa""","""Manchester United""","""1 - 2""","""23/24"""
2,1,"""01/10/2023 12:00""","""Walton Hall Park""","""Everton""","""Brighton & Hove Albion""","""1 - 2""","""23/24"""
3,1,"""01/10/2023 13:00""","""Ashton Gate Stadium""","""Bristol City""","""Leicester City""","""2 - 4""","""23/24"""
4,1,"""01/10/2023 13:00""","""Emirates Stadium""","""Arsenal""","""Liverpool""","""0 - 1""","""23/24"""
5,1,"""01/10/2023 14:00""","""Chigwell Construction Stadium""","""West Ham United""","""Manchester City""","""0 - 2""","""23/24"""


In [5]:
# Manually categorise each stadium by exporting the unique stadiums within the column
home_stadium = data.select(['Home Team', "Location"]).unique().sort(by="Home Team")
home_stadium.write_csv("team_stadiums.txt")

In [6]:
# AI agent that gets location data for each stadium
from haystack_ai.agent import stadium_location_agent

agent = False

if agent:
    len_stadiums = len(data.select("Location").unique())
    stadium_location_agent(count=len_stadiums, stadium_file="team_stadiums.txt", model="qwen2.5:7b", location_file="team_stadium_locations.csv")

In [7]:
# Convert the csv of the stadiums and their geographical location into a dictionary where the stadium is the key and the location is the value
with open("team_stadium_locations.csv", "r") as file:
    reader = csv.DictReader(file)
    stadium_region = {}
    stadium_longitude = {}
    stadium_latitude = {}

    for row in reader:
        stadium_region[row["Stadium"]] = row["Region"]
        stadium_longitude[row["Stadium"]] = row["Longitude"]
        stadium_latitude[row["Stadium"]] = row["Latitude"]

    if len(stadium_region) != len(stadium_longitude) != len(stadium_latitude):
        print(
            "Incorrect",
            len(stadium_region),
            len(stadium_longitude),
            len(stadium_latitude),
            data['Location'].unique().shape[0],
            sep='\n'
        )

In [8]:
data_new = (
    data.clone().with_columns(
        pl.col("Result").str.replace_all(" ", ""),  # Remove all whitespace
        pl.col("Date").str.to_datetime("%d/%m/%Y %H:%M"),   # Convert to DateTime data type
        pl.col("Location").alias("Stadium"),    # Rename the column
    )

    .with_columns(
        # Get results specific for home and away teams
        pl.col("Result").str.split("-").list.get(0).cast(pl.Int8).alias("Home Goals"),
        pl.col("Result").str.split("-").list.get(1).cast(pl.Int8).alias("Away Goals"),

        # Separate the date into individual variables
        pl.col("Date").dt.day().alias("Day"),
        pl.col("Date").dt.month().alias("Month"),
        pl.col("Date").dt.year().alias("Year"),
        pl.col("Date").dt.hour().alias("Hour"),

        # Identify Kickoff period based on the Date
        pl.when(pl.col("Date").dt.hour().le(13))    # <=12pm for morning
        .then(pl.lit("Morning"))
        .when(pl.col("Date").dt.hour().is_between(12, 18, closed="none"))   # 13 < time < 18 for afternoon
        .then(pl.lit("Afternoon"))
        .when(pl.col("Date").dt.hour().ge(18))  # >=18 for evening
        .then(pl.lit("Evening"))
        .alias("Kickoff"),

        # Identify geographical region of each stadium
        pl.col("Stadium").replace(stadium_region).alias("Region"),
        pl.col("Stadium").replace(stadium_longitude).alias("Longitude"),
        pl.col("Stadium").replace(stadium_latitude).alias("Latitude"),
    )

    .with_columns(
        # Calculate the goal difference
        (pl.col("Home Goals") - pl.col("Away Goals")).alias("Goal Difference"),

        # Calculate the number of goals scored
        (pl.col("Home Goals") + pl.col("Away Goals")).alias("Goals Scored"),

        # Determine the winners
        pl.when(pl.col("Home Goals").gt(pl.col("Away Goals")))
        .then(pl.lit("Home"))
        .when(pl.col("Away Goals").gt(pl.col("Home Goals")))
        .then(pl.lit("Away"))
        .otherwise(pl.lit("Draw"))
        .alias("Winner"),
    )

    .with_columns(
        # Identify winning teams
        pl.when(pl.col("Winner").eq("Away"))
        .then(pl.col("Away Team"))
        .when(pl.col("Winner").eq("Home"))
        .then(pl.col("Home Team"))
        .otherwise(pl.lit("Draw"))
        .alias("Winning Team"),

        # Identify losing teams
        pl.when(pl.col("Winner").eq("Away"))
        .then(pl.col("Home Team"))
        .when(pl.col("Winner").eq("Home"))
        .then(pl.col("Away Team"))
        .otherwise(pl.lit("Draw"))
        .alias("Losing Team"),
    )

    # Drop unnecessary columns
    .drop(["Location", "Date", "Result"])

    # Assign the right data type for a column - lower-precision variants are more memory-efficient
    .cast({"Round Number": pl.UInt8, "Day": pl.UInt8, "Month": pl.UInt8, "Year": pl.UInt16, "Hour": pl.UInt8, "Longitude": pl.Float16, "Latitude": pl.Float16, "Home Goals": pl.UInt8, "Away Goals": pl.UInt8, "Goals Scored": pl.UInt8})
)

In [9]:
# Order categories within a categorical variable
kickoff_enum = pl.Enum(["Morning", "Afternoon", "Evening"]) # Enum is an ordered categorical data type
data_new = data_new.cast({"Kickoff": kickoff_enum})

Data types: https://pola.rs/posts/understanding-polars-data-types/

`Enum` data type: https://docs.pola.rs/user-guide/expressions/categorical-data-and-enums/#enum-vs-categorical

In [10]:
# Reorder columns
df = data_new.clone().select(["Season", "Round Number", "Hour", "Day", "Month", "Year", "Kickoff", "Stadium", "Region", "Longitude", "Latitude", "Home Team", "Away Team", "Home Goals", "Away Goals", "Winner", "Winning Team", "Losing Team", "Goal Difference", "Goals Scored"])

In [11]:
df.show(limit=5)

Season,Round Number,Hour,Day,Month,Year,Kickoff,Stadium,Region,Longitude,Latitude,Home Team,Away Team,Home Goals,Away Goals,Winner,Winning Team,Losing Team,Goal Difference,Goals Scored
str,u8,u8,u8,u8,u16,enum,str,str,f16,f16,str,str,u8,u8,str,str,str,i8,u8
"""23/24""",1,11,1,10,2023,"""Morning""","""Villa Park""","""West Midlands""",-1.884766,52.5,"""Aston Villa""","""Manchester United""",1,2,"""Away""","""Manchester United""","""Aston Villa""",-1,3
"""23/24""",1,12,1,10,2023,"""Morning""","""Walton Hall Park""","""North West""",-2.951172,53.4375,"""Everton""","""Brighton & Hove Albion""",1,2,"""Away""","""Brighton & Hove Albion""","""Everton""",-1,3
"""23/24""",1,13,1,10,2023,"""Morning""","""Ashton Gate Stadium""","""South West""",-2.621094,51.4375,"""Bristol City""","""Leicester City""",2,4,"""Away""","""Leicester City""","""Bristol City""",-2,6
"""23/24""",1,13,1,10,2023,"""Morning""","""Emirates Stadium""","""London""",-0.109131,51.5625,"""Arsenal""","""Liverpool""",0,1,"""Away""","""Liverpool""","""Arsenal""",-1,1
"""23/24""",1,14,1,10,2023,"""Afternoon""","""Chigwell Construction Stadium""","""London""",-0.016449,51.53125,"""West Ham United""","""Manchester City""",0,2,"""Away""","""Manchester City""","""West Ham United""",-2,2


# Data Visualisations

In [48]:
import polars.selectors as cs
import plotly.express as px
import plotly.graph_objects as go

## Stadium Map

In [13]:
df.clone().select("Home Team", "Stadium").unpivot().group_by(pl.all()).len().rename({"value": "Team/Stadium", "len": "Number of Matches"})

variable,Team/Stadium,Number of Matches
str,str,u32
"""Stadium""","""Goodison Park""",3
"""Stadium""","""Old Trafford""",5
"""Home Team""","""West Ham United""",22
"""Stadium""","""Villa Park""",16
"""Stadium""","""Emirates Stadium""",15
…,…,…
"""Stadium""","""Prenton Park""",10
"""Home Team""","""Bristol City""",11
"""Stadium""","""Etihad Stadium""",4


In [14]:
stadium_count = df["Stadium"].value_counts().sort(by="Stadium").rename({"count": "Number of Matches"})
stadium_loc_df = pl.read_csv("team_stadium_locations.csv").join(stadium_count, on=["Stadium"], how="inner")

fig = px.scatter_map(data_frame=stadium_loc_df, lat="Latitude", lon="Longitude", color="Stadium", hover_name="Stadium", size="Number of Matches", zoom=5, title="Stadium Map", subtitle="Map of the stadiums located in England sized by the number of total matches hosted in the stadium", color_discrete_sequence=px.colors.qualitative.Dark24)

fig.update_layout(
    autosize=False,
    width=800,
    height=800,
)


## Goals Scored

### Region / Stadium

In [15]:
# Bar chart comparing which region had the most goals scored
region_goals = df.clone().group_by("Region").agg(pl.col("Goals Scored").sum()).sort(by="Goals Scored", descending=True)

px.bar(data_frame=region_goals, x="Region", y="Goals Scored", title="Total Goals Scored by Region")

In [16]:
# Bar chart showing which stadium had the most goals scored
stadium_goals = df.clone().group_by(["Stadium", "Region"]).agg(pl.col("Goals Scored").sum()).sort(by="Goals Scored", descending=True)

fig = px.bar(data_frame=stadium_goals, y="Stadium", x="Goals Scored", color="Region", title="Total Goals Scored by Stadium and Region", color_discrete_sequence=px.colors.qualitative.Dark24)

fig.update_layout(
    autosize=False,
    width=1200,
    height=700,
    yaxis={"categoryorder": "total ascending"}, # Orders the bars in ascending order (bottom up)
)

### Kickoff

In [17]:
# Bar chart showing which kickoff had the most goals scored
kickoff_goals = df.clone().group_by("Kickoff").agg(pl.col("Goals Scored").sum()).sort(by="Kickoff")

px.bar(data_frame=kickoff_goals, x="Kickoff", y="Goals Scored", title="Total Goals Scored by Kickoff")

In [18]:
# Number of games per kickoff
kickoff_games = df.clone().select(pl.col("Kickoff").value_counts()).unnest("Kickoff").sort(by="Kickoff")

fig = px.bar(data_frame=kickoff_games, x="Kickoff", y="count", title="Total Games by Kickoff")

fig.update_layout(
    autosize=False,
    width=600,
    height=600,
)

### Home vs Away

In [19]:
# Bar chart comparing Home and Away Team scores
home_away_goals = (
    data_new.clone()
    .select(['Home Goals', 'Away Goals'])
    .select(pl.sum('Home Goals', 'Away Goals'))
    .unpivot(on=cs.numeric(), variable_name="Home/Away", value_name="Goals Scored")
)

fig = px.bar(data_frame=home_away_goals, x="Home/Away", y="Goals Scored", title="Total Goals Scored Home and Away")

fig.update_layout(
    autosize=False,
    width=600,
    height=600,
)

In [20]:
# Stacked bar chart of which team scored the most goals separated by home and away goals
home_goals = df.clone().group_by("Home Team").agg(pl.col("Home Goals").sum()).sort(by="Home Team").rename({"Home Team": "Team", "Home Goals": "Home"})
away_goals = df.clone().group_by("Away Team").agg(pl.col("Away Goals").sum()).sort(by="Away Team").rename({"Away Team": "Team", "Away Goals": "Away"})
home_away_team_goals = home_goals.join(away_goals, on=['Team'], how='inner').unpivot(on=cs.numeric(), index="Team", variable_name="Home/Away", value_name="Goals Scored")

fig = px.bar(data_frame=home_away_team_goals, x="Team", y="Goals Scored", color="Home/Away", title="Total Goals Scored Home and Away by Team", color_discrete_sequence=px.colors.qualitative.Bold)

fig.update_layout(
    autosize=False,
    width=1000,
    height=600,
)

### Teams

In [21]:
total_home_away_goals = df.clone().group_by("Home Team").agg(pl.col("Home Goals", "Away Goals").sum()).with_columns((pl.col("Home Goals") + pl.col("Away Goals")).alias("Total Goals"), pl.lit("Total").alias("Season")).rename({"Home Team": "Team"}).select('Season', 'Team', 'Home Goals', 'Away Goals', 'Total Goals')

season_goals = df.clone().group_by("Season", "Home Team").agg(pl.col("Home Goals", "Away Goals").sum()).with_columns((pl.col("Home Goals") + pl.col("Away Goals")).alias("Total Goals")).sort(by="Season").rename({"Home Team": "Team"})

season_total_goals = pl.concat([season_goals, total_home_away_goals])

In [22]:
# Pie chart of goals scored segmented by teams and the season

def team_goals_by_season(season: str = "23/24", home_away: str = "Total Goals"):
    df1 = season_total_goals.filter(pl.col("Season").eq(f"{season}"))

    fig = px.pie(df1, names="Team", values=f"{home_away}", title=f"Proportion of {home_away} Scored by Teams ({season})")

    fig.update_layout(
        autosize=False,
        width=750,
        height=500,
    )

    return fig

team_goals_by_season(season="23/24", home_away="Away Goals")

## Time-Series

In [23]:
# Time series of goals scored compared between each season and the over all seasons
df_season_goals = df.clone().group_by("Season", "Round Number").agg(pl.col("Goals Scored").sum()).sort(by="Round Number")
df_total_goals = df.clone().group_by("Round Number").agg(pl.col("Goals Scored").sum()).sort(by="Round Number").with_columns(pl.lit("Total").alias("Season")).select("Season", "Round Number", "Goals Scored")
df_goals = pl.concat([df_season_goals, df_total_goals])

fig = px.line(df_goals, x="Round Number", y="Goals Scored", color="Season", color_discrete_sequence=px.colors.qualitative.G10)

fig.update_layout(
    autosize=False,
    width=1300,
    height=600,
    title="Time-series of Goals Score by Round Number",
    xaxis = dict(
        tickmode = 'linear',
        tick0 = 1,
        dtick = 1
    )
)



In [24]:
# Time series of the number of goals each team has scored over the season

def track_team_goals(team: str = "Arsenal"):

    team_season_goals = df.clone().filter((pl.col("Home Team") == f"{team}") | (pl.col("Away Team") == f"{team}")).with_columns(
        pl.when(pl.col("Home Team") == f"{team}")
        .then(pl.col("Home Goals"))
        .when(pl.col("Away Team") == f"{team}")
        .then(pl.col("Away Goals"))
        .alias("Goal Tracking")
    ).select("Season", "Round Number", "Goal Tracking")

    total_team_season_goals = team_season_goals.clone().group_by("Round Number").agg(pl.col("Goal Tracking").sum()).sort(by="Round Number").with_columns(pl.lit("Total").alias("Season")).select("Season", "Round Number", "Goal Tracking").cast({"Goal Tracking" : pl.UInt8})

    team_season_goals_df = pl.concat([team_season_goals, total_team_season_goals])

    fig = px.line(team_season_goals_df, x="Round Number", y="Goal Tracking", color="Season", title=f"Goals Scored by {team} per Round")

    fig.update_layout(
        autosize=False,
        width=1300,
        height=600,
        xaxis = dict(tick0=1, dtick=1),
    )

    return fig

track_team_goals()

In [64]:
# Bar chart showing most goals scored by a single team at each round for each season

def most_goals_per_round_season(season: str = "23/24"):
    df1 = df.group_by('Season', 'Round Number').agg(pl.all().sort_by('Goals Scored').last()).sort(by="Round Number").select("Season", "Round Number", "Winning Team", "Goals Scored").filter(pl.col("Season").eq(season))

    fig = go.Figure()

    fig.add_bar(x=df1['Round Number'].to_list(), y=df1['Goals Scored'].to_list(), text=df1['Winning Team'].to_list(), textposition='inside', textangle=-90, insidetextanchor="middle", textfont_size=12,)

    fig.update_layout(
        title=f"Most Goals Scored in Each Round for the {season} Season",
        xaxis=dict(tickmode='linear', tick0=1, dtick=1),
    )

    return fig

most_goals_per_round_season(season="24/25")


In [72]:
# Source - https://stackoverflow.com/a/72019719

most_goals_per_round_season = df.group_by('Season', 'Round Number').agg(pl.all().sort_by('Goals Scored').last()).sort(by="Round Number").select("Season", "Round Number", "Goals Scored")
most_goals_per_round_total = most_goals_per_round_season.clone().group_by("Round Number").agg(pl.col("Goals Scored").sum()).with_columns(pl.lit("Total").alias("Season")).select("Season", "Round Number", "Goals Scored").cast({"Goals Scored" : pl.UInt8})
most_goals_per_round = pl.concat([most_goals_per_round_season, most_goals_per_round_total]).sort(by="Round Number")

fig = px.line(data_frame=most_goals_per_round, x="Round Number", y="Goals Scored", color="Season", title="Most Goals Scored per Round")

fig.update_layout(
    xaxis=dict(tickmode='linear', tick0=1, dtick=1),
)